In [1]:
import os
import pandas as pd
import numpy as np
import plotly.express as px

import seaborn as sns
import matplotlib.pyplot as plt

/Users/phuchoangle/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/phuchoangle/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


In [2]:
layer1 = pd.read_csv('/Users/phuchoangle/Desktop/concero-series-a/layer/layer1.csv')
layer2 = pd.read_csv('/Users/phuchoangle/Desktop/concero-series-a/layer/layer2.csv')
layer3 = pd.read_csv('/Users/phuchoangle/Desktop/concero-series-a/layer/layer3.csv')

layerzero = pd.read_csv('/Users/phuchoangle/Desktop/concero-series-a/layer/layerzero.csv')
wormhole = pd.read_csv('/Users/phuchoangle/Desktop/concero-series-a/layer/wormhole.csv')
ccip = pd.read_csv('/Users/phuchoangle/Desktop/concero-series-a/layer/chainlink-ccip.csv')
axelar = pd.read_csv('/Users/phuchoangle/Desktop/concero-series-a/layer/axelar.csv')
hyperlane = pd.read_csv('/Users/phuchoangle/Desktop/concero-series-a/layer/hyperlane.csv')

In [5]:
# Count unique chains in each layer
total_layer1 = layer1['Name'].nunique()
total_layer2 = layer2['Name'].nunique()
total_layer3 = layer3['Name'].nunique()

print(f"Total Layer 1 chains: {total_layer1}")
print(f"Total Layer 2 chains: {total_layer2}")
print(f"Total Layer 3 chains: {total_layer3}")
print(f"Grand total: {total_layer1 + total_layer2 + total_layer3}")

Total Layer 1 chains: 338
Total Layer 2 chains: 131
Total Layer 3 chains: 29
Grand total: 498


In [9]:
import pandas as pd

# Load the data
layer1 = pd.read_csv('/Users/phuchoangle/Desktop/concero-series-a/layer/layer1.csv')
layer2 = pd.read_csv('/Users/phuchoangle/Desktop/concero-series-a/layer/layer2.csv')
layer3 = pd.read_csv('/Users/phuchoangle/Desktop/concero-series-a/layer/layer3.csv')

# Combine all layers and filter for 2018-2025
all_chains = pd.concat([layer1, layer2, layer3])
all_chains = all_chains[(all_chains['Year'] >= 2018) & (all_chains['Year'] <= 2025)]

# Calculate total number of chains in this period
total_chains = len(all_chains)

# Calculate total number of quarters (2018 Q1 to 2025 Q4 = 8 years * 4 quarters = 32 quarters)
total_quarters = (2025 - 2018 + 1) * 4

# Calculate average new chains per quarter
average_new_chains = total_chains / total_quarters

print(f"Total new chains (2018-2025): {total_chains}")
print(f"Total quarters (2018 Q1 - 2025 Q4): {total_quarters}")
print(f"Average new chains per quarter: {average_new_chains:.2f}")

Total new chains (2018-2025): 476
Total quarters (2018 Q1 - 2025 Q4): 32
Average new chains per quarter: 14.88


In [10]:
import pandas as pd
import numpy as np
from scipy import stats

# Load the data
layer1 = pd.read_csv('/Users/phuchoangle/Desktop/concero-series-a/layer/layer1.csv')
layer2 = pd.read_csv('/Users/phuchoangle/Desktop/concero-series-a/layer/layer2.csv')
layer3 = pd.read_csv('/Users/phuchoangle/Desktop/concero-series-a/layer/layer3.csv')

# Combine all layers and filter for 2018-2025
all_chains = pd.concat([layer1, layer2, layer3])
all_chains = all_chains[(all_chains['Year'] >= 2018) & (all_chains['Year'] <= 2025)]

# Count chains per year
yearly_counts = all_chains['Year'].value_counts().sort_index()

# Calculate quarters since 2018
yearly_counts = yearly_counts.reset_index()
yearly_counts.columns = ['Year', 'Count']

# Calculate quarters since 2018 Q1 (0 = 2018 Q1, 1 = 2018 Q2, ..., 31 = 2025 Q4)
yearly_counts['Quarters_Since_2018'] = (yearly_counts['Year'] - 2018) * 4

# Distribute yearly counts to quarters (simple linear interpolation)
quarterly_counts = []
for _, row in yearly_counts.iterrows():
    base_quarter = row['Quarters_Since_2018']
    for q in range(4):
        quarter = base_quarter + q
        quarterly_counts.append({
            'Quarter_Number': quarter,
            'Year': row['Year'],
            'Quarter': q + 1,
            'Chains': row['Count'] / 4  # Distribute evenly
        })

quarterly_df = pd.DataFrame(quarterly_counts)

# Calculate linear regression
x = quarterly_df['Quarter_Number']
y = quarterly_df['Chains']
slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)

# Calculate projected chains per quarter
start_quarter = 0  # 2018 Q1
end_quarter = 31    # 2025 Q4
start_chains = intercept + slope * start_quarter
end_chains = intercept + slope * end_quarter
total_quarters = end_quarter - start_quarter + 1

# Calculate total chains using linear growth
# This is the area under the line: total = n/2 * (a1 + an)
total_chains_linear = (total_quarters / 2) * (start_chains + end_chains)
average_linear = total_chains_linear / total_quarters

# Compare with simple average
simple_average = quarterly_df['Chains'].mean()

print(f"Linear growth model (2018 Q1 - 2025 Q4):")
print(f"Starting chains/quarter (2018 Q1): {start_chains:.2f}")
print(f"Ending chains/quarter (2025 Q4): {end_chains:.2f}")
print(f"Average chains/quarter (linear growth): {average_linear:.2f}")
print(f"\nFor comparison:")
print(f"Simple average (equal distribution): {simple_average:.2f}")
print(f"Difference: {average_linear - simple_average:+.2f} chains/quarter")
print(f"Growth rate: {slope:.2f} chains/quarter (linear trend)")

# Calculate percentage difference
if simple_average != 0:
    pct_difference = ((average_linear - simple_average) / simple_average) * 100
    print(f"\nThe linear growth model estimates are {abs(pct_difference):.1f}% {'higher' if pct_difference > 0 else 'lower'} than the simple average.")

Linear growth model (2018 Q1 - 2025 Q4):
Starting chains/quarter (2018 Q1): 1.49
Ending chains/quarter (2025 Q4): 28.26
Average chains/quarter (linear growth): 14.88

For comparison:
Simple average (equal distribution): 14.88
Difference: +0.00 chains/quarter
Growth rate: 0.86 chains/quarter (linear trend)

The linear growth model estimates are 0.0% lower than the simple average.


In [3]:
year_counts_layer1 = layer1['Year'].value_counts().sort_index()
year_counts_layer2 = layer2['Year'].value_counts().sort_index()
year_counts_layer3 = layer3['Year'].value_counts().sort_index()

In [4]:
# Assuming you have these counts already
year_counts_layer1 = layer1['Year'].value_counts().sort_index()
year_counts_layer2 = layer2['Year'].value_counts().sort_index()
year_counts_layer3 = layer3['Year'].value_counts().sort_index()

# Create a DataFrame with all years and counts
df_combined = pd.DataFrame({
    'Layer 1': year_counts_layer1,
    'Layer 2': year_counts_layer2,
    'Layer 3': year_counts_layer3
}).fillna(0)  # Fill NaN values with 0

# Sort by index (year)
df_combined = df_combined.sort_index()

# Add a total column if you want
df_combined['Total'] = df_combined.sum(axis=1)

print(df_combined)

      Layer 1  Layer 2  Layer 3  Total
Year                                  
2013        4      0.0      0.0    4.0
2014        4      0.0      0.0    4.0
2015        2      0.0      0.0    2.0
2016        3      0.0      0.0    3.0
2017        9      0.0      0.0    9.0
2018       21      0.0      0.0   21.0
2019       21      0.0      0.0   21.0
2020       23      2.0      0.0   25.0
2021       36      9.0      0.0   45.0
2022       43      5.0      0.0   48.0
2023       59     21.0      2.0   82.0
2024       86     81.0     24.0  191.0
2025       27     13.0      3.0   43.0


In [5]:
import plotly.express as px

# Create the melted dataframe with all years
df_melted = df_combined.reset_index(names=['Year']).melt(
    id_vars=['Year'], 
    value_vars=['Layer 1', 'Layer 2', 'Layer 3'],
    var_name='Layer Type',
    value_name='Count'
)

# Create grouped bar chart
fig = px.bar(df_melted, 
             x='Year', 
             y='Count',
             color='Layer Type',
             title='New Blockchain Platforms Launched by Year and Layer (2013-2025)',
             labels={'Year': 'Year', 
                    'Count': 'Number of New Platforms',
                    'Layer Type': 'Layer'},
             barmode='group',
             color_discrete_sequence=['#36D7B7', '#FFB74D', '#FF7676'])

# Update layout with explicit x-axis settings
fig.update_layout(
    plot_bgcolor='rgb(17,17,17)',
    paper_bgcolor='rgb(17,17,17)',
    font=dict(color='white'),
    xaxis=dict(
        gridcolor='rgba(128,128,128,0.2)',
        tickcolor='white',
        showgrid=False,
        range=[2012.5, 2025.5],  # Extended range to show all years
        dtick=1,  # Show every year
        tickmode='linear'  # Ensure all years are shown
    ),
    yaxis=dict(
        gridcolor='rgba(128,128,128,0.2)',
        tickcolor='white',
        showgrid=True,
        gridwidth=1
    ),
    legend=dict(
        bgcolor='rgba(0,0,0,0)',
        font=dict(color='white')
    ),
    bargap=0.15,
    bargroupgap=0.1
)

# Update axes
fig.update_xaxes(tickangle=0)
fig.update_yaxes(title_standoff=20)

# Show the plot
fig.show()

In [6]:
import plotly.express as px
import pandas as pd

# Calculate cumulative sums for each layer
df_cumulative = df_combined.copy()
df_cumulative['Layer 1'] = df_cumulative['Layer 1'].cumsum()
df_cumulative['Layer 2'] = df_cumulative['Layer 2'].cumsum()
df_cumulative['Layer 3'] = df_cumulative['Layer 3'].cumsum()

# Melt the dataframe for plotting
df_melted = df_cumulative.reset_index(names=['Year']).melt(
    id_vars=['Year'],
    value_vars=['Layer 1', 'Layer 2', 'Layer 3'],
    var_name='Layer Type',
    value_name='Total Number of Platforms'
)

# Create line plot
fig = px.line(df_melted,
              x='Year',
              y='Total Number of Platforms',
              color='Layer Type',
              title='Cumulative Growth of Blockchain Platforms by Layer (2013-2025)',
              labels={'Year': 'Year',
                     'Total Number of Platforms': 'Total Number of Platforms',
                     'Layer Type': 'Layer'},
              color_discrete_sequence=['#36D7B7', '#FFB74D', '#FF7676'])

# Update layout
fig.update_layout(
    plot_bgcolor='rgb(17,17,17)',
    paper_bgcolor='rgb(17,17,17)',
    font=dict(color='white'),
    xaxis=dict(
        gridcolor='rgba(128,128,128,0.2)',
        tickcolor='white',
        showgrid=False,
        range=[2012.5, 2025.5],
        dtick=1,
        tickmode='linear'
    ),
    yaxis=dict(
        gridcolor='rgba(128,128,128,0.2)',
        tickcolor='white',
        showgrid=True,
        gridwidth=1
    ),
    legend=dict(
        bgcolor='rgba(0,0,0,0)',
        font=dict(color='white')
    )
)

# Update line properties
fig.update_traces(
    mode='lines',  # Use lines only
    line=dict(width=2)  # Adjust line width
)

# Show the plot
fig.show()

In [7]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# First, let's combine the blockchain layer data
year_counts_layer1 = layer1['Year'].value_counts().sort_index()
year_counts_layer2 = layer2['Year'].value_counts().sort_index()
year_counts_layer3 = layer3['Year'].value_counts().sort_index()

# Create cumulative blockchain counts
df_combined = pd.DataFrame({
    'Layer 1': year_counts_layer1,
    'Layer 2': year_counts_layer2,
    'Layer 3': year_counts_layer3
}).fillna(0)

df_combined = df_combined.sort_index()
df_combined['Total New Chains'] = df_combined.sum(axis=1)
df_combined['Cumulative Total Chains'] = df_combined['Total New Chains'].cumsum()

# Process interoperability data
def process_interop_data(df, protocol_name):
    return pd.DataFrame({
        'Year': df['Year'],
        'Protocol': protocol_name,
        'Count': 1
    })

axelar_processed = process_interop_data(axelar, 'Axelar')
ccip_processed = process_interop_data(ccip, 'Chainlink CCIP')
layerzero_processed = process_interop_data(layerzero, 'LayerZero')
wormhole_processed = process_interop_data(wormhole, 'Wormhole')

# Combine all interop data
interop_combined = pd.concat([
    axelar_processed,
    ccip_processed,
    layerzero_processed,
    wormhole_processed
])

# Get yearly integration counts
yearly_integrations = interop_combined.groupby('Year')['Count'].sum()
cumulative_integrations = yearly_integrations.cumsum()

# Create the plot
fig = go.Figure()

# Add cumulative blockchain count
fig.add_trace(go.Scatter(
    x=df_combined.index,
    y=df_combined['Cumulative Total Chains'],
    name='Total Blockchain Platforms',
    line=dict(color='#36D7B7', width=3),
    mode='lines'
))

# Add cumulative integrations
fig.add_trace(go.Scatter(
    x=cumulative_integrations.index,
    y=cumulative_integrations,
    name='Total Protocol Integrations',
    line=dict(color='#FF7676', width=3),
    mode='lines'
))

# Update layout
fig.update_layout(
    title='Growth of Blockchain Platforms vs Protocol Integrations (2013-2025)',
    xaxis_title='Year',
    yaxis_title='Count',
    plot_bgcolor='rgb(17,17,17)',
    paper_bgcolor='rgb(17,17,17)',
    font=dict(color='white'),
    showlegend=True,
    legend=dict(
        bgcolor='rgba(0,0,0,0)',
        font=dict(color='white')
    ),
    xaxis=dict(
        gridcolor='rgba(128,128,128,0.2)',
        tickcolor='white',
        showgrid=False,
        range=[2013, 2025],
        dtick=1
    ),
    yaxis=dict(
        gridcolor='rgba(128,128,128,0.2)',
        tickcolor='white',
        showgrid=True,
        gridwidth=1
    )
)

# Add shaded area between the lines to highlight the gap
fig.add_trace(go.Scatter(
    x=df_combined.index,
    y=df_combined['Cumulative Total Chains'],
    fill='tonexty',
    fillcolor='rgba(54,215,183,0.1)',
    line=dict(width=0),
    showlegend=False,
    name='Gap'
))

fig.show()

In [8]:
import plotly.graph_objects as go

years = df_combined.index

# Bar traces for new platforms per year
bar_layer1 = go.Bar(
    x=years, y=df_combined['Layer 1'],
    name='Layer 1 (New)', marker_color='#36D7B7', opacity=0.5
)
bar_layer2 = go.Bar(
    x=years, y=df_combined['Layer 2'],
    name='Layer 2 (New)', marker_color='#FFB74D', opacity=0.5
)
bar_layer3 = go.Bar(
    x=years, y=df_combined['Layer 3'],
    name='Layer 3 (New)', marker_color='#FF7676', opacity=0.5
)

# Cumulative lines
cum_layer1 = go.Scatter(
    x=years, y=df_combined['Layer 1'].cumsum(),
    name='Layer 1 (Cumulative)', mode='lines+markers',
    line=dict(color='#36D7B7', width=4, dash='solid'),
    yaxis='y2'
)
cum_layer2 = go.Scatter(
    x=years, y=df_combined['Layer 2'].cumsum(),
    name='Layer 2 (Cumulative)', mode='lines+markers',
    line=dict(color='#FFB74D', width=4, dash='solid'),
    yaxis='y2'
)
cum_layer3 = go.Scatter(
    x=years, y=df_combined['Layer 3'].cumsum(),
    name='Layer 3 (Cumulative)', mode='lines+markers',
    line=dict(color='#FF7676', width=4, dash='solid'),
    yaxis='y2'
)

layout = go.Layout(
    title='New Blockchain Platforms and Cumulative Growth by Layer (2013-2025)',
    plot_bgcolor='rgb(17,17,17)',
    paper_bgcolor='rgb(17,17,17)',
    font=dict(color='white'),
    barmode='group',  # or 'stack' if you want stacked bars
    xaxis=dict(
        title='Year',
        dtick=2,  # Show every 2 years
        tickmode='linear',
        gridcolor='rgba(128,128,128,0.2)',
        tickcolor='white',
        range=[2012.5, 2025.5]
    ),
    yaxis=dict(
        title='Number of New Platforms',
        gridcolor='rgba(128,128,128,0.2)',
        tickcolor='white',
        showgrid=True,
        gridwidth=1
    ),
    yaxis2=dict(
        title='Cumulative Platforms',
        overlaying='y',
        side='right',
        showgrid=False
    ),
    legend=dict(
        bgcolor='rgba(0,0,0,0)',
        font=dict(color='white'),
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1
    ),
    margin=dict(l=60, r=60, t=80, b=60)
)

fig = go.Figure(data=[bar_layer1, bar_layer2, bar_layer3, cum_layer1, cum_layer2, cum_layer3], layout=layout)
fig.show()

In [9]:
import plotly.io as pio

years = df_combined.index

# Stacked bars for new platforms per year
bar1 = go.Bar(
    x=years,
    y=df_combined['Layer 1'],
    name='Layer 1',
    marker_color='#36D7B7'
)
bar2 = go.Bar(
    x=years,
    y=df_combined['Layer 2'],
    name='Layer 2',
    marker_color='#FFB74D'
)
bar3 = go.Bar(
    x=years,
    y=df_combined['Layer 3'],
    name='Layer 3',
    marker_color='#FF7676'
)

# Cumulative total line
cum_line = go.Scatter(
    x=years,
    y=df_combined[['Layer 1', 'Layer 2', 'Layer 3']].sum(axis=1).cumsum(),
    name='Cumulative Total',
    mode='lines+markers',
    line=dict(color='white', width=4),
    yaxis='y2'
)

# Calculate max values for each axis for proper scaling
max_new = df_combined[['Layer 1', 'Layer 2', 'Layer 3']].sum(axis=1).max()
max_cum = df_combined[['Layer 1', 'Layer 2', 'Layer 3']].sum(axis=1).cumsum().max()

layout = go.Layout(
    title='New Blockchain Platforms by Year (Stacked) and Cumulative Total (Line)',
    plot_bgcolor='rgb(17,17,17)',
    paper_bgcolor='rgb(17,17,17)',
    font=dict(color='white'),
    barmode='stack',
    xaxis=dict(
        title='Year',
        dtick=2,
        tickmode='linear',
        gridcolor='rgba(128,128,128,0.2)',
        tickcolor='white',
        range=[2012.5, 2025.5],
        showline=True,
        showgrid=True,
        zeroline=False
    ),
    yaxis=dict(
        title='Number of New Platforms',
        gridcolor='rgba(128,128,128,0.2)',
        tickcolor='white',
        showgrid=True,
        gridwidth=1,
        range=[0, max_new * 1.1],  # Dynamic scaling with padding
        showline=True,
        zeroline=True,
        zerolinecolor='rgba(255,255,255,0.8)',
        zerolinewidth=2
    ),
    yaxis2=dict(
        title='Cumulative Platforms',
        overlaying='y',
        side='right',
        showgrid=False,
        showline=True,
        zeroline=False,  # Don't show second zeroline
        range=[0, max_cum * 1.1],  # Dynamic scaling with padding
        anchor="x"
    ),
    legend=dict(
        bgcolor='rgba(0,0,0,0)',
        font=dict(color='white'),
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1
    ),
    margin=dict(l=60, r=60, t=80, b=60)
)

# Create the figure with all traces
fig = go.Figure(data=[bar1, bar2, bar3, cum_line], layout=layout)

# Force the zero baseline alignment with an update
fig.update_layout(
    yaxis=dict(
        scaleanchor="y2",
        scaleratio=max_cum/max_new,
        constraintoward="bottom"
    )
)

fig.show()

In [10]:
import pandas as pd
import plotly.graph_objects as go

# First, let's combine the blockchain layer data as before
year_counts_layer1 = layer1['Year'].value_counts().sort_index()
year_counts_layer2 = layer2['Year'].value_counts().sort_index()
year_counts_layer3 = layer3['Year'].value_counts().sort_index()

# Create cumulative blockchain counts
df_combined = pd.DataFrame({
    'Layer 1': year_counts_layer1,
    'Layer 2': year_counts_layer2,
    'Layer 3': year_counts_layer3
}).fillna(0)

df_combined = df_combined.sort_index()
df_combined['Total New Chains'] = df_combined.sum(axis=1)
df_combined['Cumulative Total Chains'] = df_combined['Total New Chains'].cumsum()

# Process each protocol's data separately
def get_protocol_cumulative(df):
    yearly_counts = df['Year'].value_counts().sort_index()
    return yearly_counts.cumsum()

layerzero_cumulative = get_protocol_cumulative(layerzero)
hyperlane_cumulative = get_protocol_cumulative(hyperlane)
wormhole_cumulative = get_protocol_cumulative(wormhole)

# Create the plot
fig = go.Figure()

# Add cumulative blockchain count
fig.add_trace(go.Scatter(
    x=df_combined.index,
    y=df_combined['Cumulative Total Chains'],
    name='Total Blockchain Platforms',
    line=dict(color='#36D7B7', width=4),
    mode='lines'
))

# Add each protocol's cumulative integrations
fig.add_trace(go.Scatter(
    x=layerzero_cumulative.index,
    y=layerzero_cumulative,
    name='LayerZero Integrations',
    line=dict(color='#FF7676', width=2),
    mode='lines'
))

fig.add_trace(go.Scatter(
    x=hyperlane_cumulative.index,
    y=hyperlane_cumulative,
    name='Hyperlane Integrations',
    line=dict(color='#FFB74D', width=2),
    mode='lines'
))

fig.add_trace(go.Scatter(
    x=wormhole_cumulative.index,
    y=wormhole_cumulative,
    name='Wormhole Integrations',
    line=dict(color='#B39DDB', width=2),
    mode='lines'
))

# Update layout
fig.update_layout(
    title='Growth of Blockchain Platforms vs Individual Protocol Integrations (2013-2025)',
    xaxis_title='Year',
    yaxis_title='Cumulative Count',
    plot_bgcolor='rgb(17,17,17)',
    paper_bgcolor='rgb(17,17,17)',
    font=dict(color='white'),
    showlegend=True,
    legend=dict(
        bgcolor='rgba(0,0,0,0)',
        font=dict(color='white'),
        y=0.99,
        x=0.01,
        bordercolor='white',
        borderwidth=1
    ),
    xaxis=dict(
        gridcolor='rgba(128,128,128,0.2)',
        tickcolor='white',
        showgrid=False,
        range=[2013, 2025],
        dtick=1
    ),
    yaxis=dict(
        gridcolor='rgba(128,128,128,0.2)',
        tickcolor='white',
        showgrid=True,
        gridwidth=1
    ),
    hovermode='x unified'
)

# Add shaded area between total platforms and highest protocol line
max_protocol = pd.concat([
    layerzero_cumulative,
    hyperlane_cumulative,
    wormhole_cumulative
]).groupby(level=0).max()

fig.add_trace(go.Scatter(
    x=df_combined.index,
    y=df_combined['Cumulative Total Chains'],
    fill='tonexty',
    fillcolor='rgba(54,215,183,0.1)',
    line=dict(width=0),
    showlegend=False,
    name='Integration Gap'
))

fig.show()

In [11]:
import pandas as pd
import plotly.graph_objects as go

# First, let's combine the blockchain layer data as before
year_counts_layer1 = layer1['Year'].value_counts().sort_index()
year_counts_layer2 = layer2['Year'].value_counts().sort_index()
year_counts_layer3 = layer3['Year'].value_counts().sort_index()

# Create cumulative blockchain counts
df_combined = pd.DataFrame({
    'Layer 1': year_counts_layer1,
    'Layer 2': year_counts_layer2,
    'Layer 3': year_counts_layer3
}).fillna(0)

df_combined = df_combined.sort_index()
df_combined['Total New Chains'] = df_combined.sum(axis=1)
df_combined['Cumulative Total Chains'] = df_combined['Total New Chains'].cumsum()

# Process each protocol's data separately
def get_protocol_cumulative(df):
    yearly_counts = df['Year'].value_counts().sort_index()
    return yearly_counts.cumsum()

layerzero_cumulative = get_protocol_cumulative(layerzero)
hyperlane_cumulative = get_protocol_cumulative(hyperlane)
wormhole_cumulative = get_protocol_cumulative(wormhole)

# Create the plot
fig = go.Figure()

# Add cumulative blockchain count
fig.add_trace(go.Scatter(
    x=df_combined.index,
    y=df_combined['Cumulative Total Chains'],
    name='Total Blockchain Platforms',
    line=dict(color='#36D7B7', width=4),
    mode='lines'
))

# Add each protocol's cumulative integrations
fig.add_trace(go.Scatter(
    x=layerzero_cumulative.index,
    y=layerzero_cumulative,
    name='LayerZero Integrations',
    line=dict(color='#808080', width=2),  # Changed to grey
    mode='lines'
))

fig.add_trace(go.Scatter(
    x=hyperlane_cumulative.index,
    y=hyperlane_cumulative,
    name='Hyperlane Integrations',
    line=dict(color='#0000FF', width=2),  # Changed to blue
    mode='lines'
))

fig.add_trace(go.Scatter(
    x=wormhole_cumulative.index,
    y=wormhole_cumulative,
    name='Wormhole Integrations',
    line=dict(color='#800080', width=2),  # Changed to purple
    mode='lines'
))

# Update layout
fig.update_layout(
    title='Growth of Blockchain Platforms vs Individual Protocol Integrations (2013-2025)',
    xaxis_title='Year',
    yaxis_title='Cumulative Count',
    plot_bgcolor='rgb(17,17,17)',
    paper_bgcolor='rgb(17,17,17)',
    font=dict(color='white'),
    showlegend=True,
    legend=dict(
        bgcolor='rgba(0,0,0,0)',
        font=dict(color='white'),
        y=0.99,
        x=0.01,
        bordercolor='white',
        borderwidth=1
    ),
    xaxis=dict(
        gridcolor='rgba(128,128,128,0.2)',
        tickcolor='white',
        showgrid=False,
        range=[2013, 2025],
        dtick=1
    ),
    yaxis=dict(
        gridcolor='rgba(128,128,128,0.2)',
        tickcolor='white',
        showgrid=True,
        gridwidth=1
    ),
    hovermode='x unified'
)

# Add shaded area between total platforms and highest protocol line
max_protocol = pd.concat([
    layerzero_cumulative,
    hyperlane_cumulative,
    wormhole_cumulative
]).groupby(level=0).max()

fig.add_trace(go.Scatter(
    x=df_combined.index,
    y=df_combined['Cumulative Total Chains'],
    fill='tonexty',
    fillcolor='rgba(54,215,183,0.1)',
    line=dict(width=0),
    showlegend=False,
    name='Integration Gap'
))

fig.show()

# Save as high-resolution PNG
fig.write_image("blockchain_integration_hd.png", width=1200, height=800)

In [12]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np
from scipy.optimize import curve_fit

# Function to fit exponential growth
def exp_growth(x, a, b):
    return a * np.exp(b * x)

# First, let's combine the blockchain layer data as before
year_counts_layer1 = layer1['Year'].value_counts().sort_index()
year_counts_layer2 = layer2['Year'].value_counts().sort_index()
year_counts_layer3 = layer3['Year'].value_counts().sort_index()

# Create cumulative blockchain counts
df_combined = pd.DataFrame({
    'Layer 1': year_counts_layer1,
    'Layer 2': year_counts_layer2,
    'Layer 3': year_counts_layer3
}).fillna(0)

df_combined = df_combined.sort_index()
df_combined['Total New Chains'] = df_combined.sum(axis=1)
df_combined['Cumulative Total Chains'] = df_combined['Total New Chains'].cumsum()

# Filter data from 2019
df_combined = df_combined[df_combined.index >= 2019]

# Process each protocol's data separately
def get_protocol_cumulative(df):
    yearly_counts = df['Year'].value_counts().sort_index()
    return yearly_counts.cumsum()

layerzero_cumulative = get_protocol_cumulative(layerzero)
hyperlane_cumulative = get_protocol_cumulative(hyperlane)
wormhole_cumulative = get_protocol_cumulative(wormhole)

# Create projection data
def create_projection(data, name):
    years = np.array(data.index.astype(float))
    values = np.array(data.values)
    
    # Normalize years to start from 0
    years_normalized = years - years[0]
    
    # Fit exponential curve
    popt, _ = curve_fit(exp_growth, years_normalized, values, p0=[1, 0.1])
    
    # Create projection years
    future_years = np.arange(years[-1] + 1, 2028)
    future_years_normalized = np.array(future_years - years[0])
    
    # Calculate projected values
    projected_values = exp_growth(future_years_normalized, *popt)
    
    # Combine historical and projected data
    all_years = np.concatenate([years, future_years])
    all_values = np.concatenate([values, projected_values])
    
    return pd.Series(all_values, index=all_years, name=name)

# Create projections
total_chains_proj = create_projection(df_combined['Cumulative Total Chains'], 'Total Chains')
layerzero_proj = create_projection(layerzero_cumulative, 'LayerZero')
hyperlane_proj = create_projection(hyperlane_cumulative, 'Hyperlane')
wormhole_proj = create_projection(wormhole_cumulative, 'Wormhole')

# Create the plot
fig = go.Figure()

# Add historical and projected data for total chains
fig.add_trace(go.Scatter(
    x=total_chains_proj.index,
    y=total_chains_proj.values,
    name='Total Blockchain Platforms',
    line=dict(color='#36D7B7', width=4),
    mode='lines'
))

# Add vertical line at 2025 to separate historical and projected data
fig.add_vline(x=2025, line_dash="dash", line_color="white", opacity=0.3)

# Add protocol projections
fig.add_trace(go.Scatter(
    x=layerzero_proj.index,
    y=layerzero_proj.values,
    name='LayerZero Integrations',
    line=dict(color='#808080', width=2),
    mode='lines'
))

fig.add_trace(go.Scatter(
    x=hyperlane_proj.index,
    y=hyperlane_proj.values,
    name='Hyperlane Integrations',
    line=dict(color='#0000FF', width=2),
    mode='lines'
))

fig.add_trace(go.Scatter(
    x=wormhole_proj.index,
    y=wormhole_proj.values,
    name='Wormhole Integrations',
    line=dict(color='#800080', width=2),
    mode='lines'
))

# Update layout
fig.update_layout(
    title='Projected Growth of Blockchain Platforms vs Protocol Integrations (2019-2027)',
    xaxis_title='Year',
    yaxis_title='Cumulative Count',
    plot_bgcolor='rgb(17,17,17)',
    paper_bgcolor='rgb(17,17,17)',
    font=dict(color='white'),
    showlegend=True,
    legend=dict(
        bgcolor='rgba(0,0,0,0)',
        font=dict(color='white'),
        y=0.99,
        x=0.01,
        bordercolor='white',
        borderwidth=1
    ),
    xaxis=dict(
        gridcolor='rgba(128,128,128,0.2)',
        tickcolor='white',
        showgrid=False,
        range=[2019, 2027],
        dtick=1
    ),
    yaxis=dict(
        gridcolor='rgba(128,128,128,0.2)',
        tickcolor='white',
        showgrid=True,
        gridwidth=1
    ),
    hovermode='x unified'
)

# Add shaded area
fig.add_trace(go.Scatter(
    x=total_chains_proj.index,
    y=total_chains_proj.values,
    fill='tonexty',
    fillcolor='rgba(54,215,183,0.1)',
    line=dict(width=0),
    showlegend=False,
    name='Integration Gap'
))

# Add annotation for projection start
fig.add_annotation(
    x=2025,
    y=fig.data[0].y.max()/2,
    text="Projection Start",
    showarrow=True,
    arrowhead=1,
    ax=0,
    ay=-40,
    font=dict(color='white')
)

fig.show()

# Save as high-resolution PNG
fig.write_image("blockchain_projection_hd.png", width=1200, height=800)

In [13]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np
from scipy.optimize import curve_fit

# Function to fit exponential growth
def exp_growth(x, a, b):
    return a * np.exp(b * x)

# First, let's combine the blockchain layer data
year_counts_layer1 = layer1['Year'].value_counts().sort_index()
year_counts_layer2 = layer2['Year'].value_counts().sort_index()
year_counts_layer3 = layer3['Year'].value_counts().sort_index()

# Create DataFrame with all years and counts
df_combined = pd.DataFrame({
    'Layer 1': year_counts_layer1,
    'Layer 2': year_counts_layer2,
    'Layer 3': year_counts_layer3
}).fillna(0)

df_combined = df_combined.sort_index()
df_combined['Total New Chains'] = df_combined.sum(axis=1)
df_combined['Cumulative Total Chains'] = df_combined['Total New Chains'].cumsum()

# Process each protocol's data separately
def get_protocol_cumulative(df):
    yearly_counts = df['Year'].value_counts().sort_index()
    return yearly_counts.cumsum()

layerzero_cumulative = get_protocol_cumulative(layerzero)
hyperlane_cumulative = get_protocol_cumulative(hyperlane)
wormhole_cumulative = get_protocol_cumulative(wormhole)

# Create projection data
def create_projection(data, name):
    years = np.array(data.index.astype(float))
    values = np.array(data.values)
    
    # Normalize years to start from 0
    years_normalized = years - years[0]
    
    # Fit exponential curve
    popt, _ = curve_fit(exp_growth, years_normalized, values, p0=[1, 0.1])
    
    # Create projection years
    future_years = np.arange(years[-1] + 1, 2028)
    future_years_normalized = np.array(future_years - years[0])
    
    # Calculate projected values
    projected_values = exp_growth(future_years_normalized, *popt)
    
    # Combine historical and projected data
    all_years = np.concatenate([years, future_years])
    all_values = np.concatenate([values, projected_values])
    
    return pd.Series(all_values, index=all_years, name=name)

# Create projections
total_chains_proj = create_projection(df_combined['Cumulative Total Chains'], 'Total Chains')
layerzero_proj = create_projection(layerzero_cumulative, 'LayerZero')
hyperlane_proj = create_projection(hyperlane_cumulative, 'CCIP')
wormhole_proj = create_projection(wormhole_cumulative, 'Wormhole')

# Create the combined plot
fig = go.Figure()

# Add stacked bars for new platforms by year (focus on 2019-2027)
filtered_df = df_combined.loc[2019:2027]
for layer, color in zip(['Layer 1', 'Layer 2', 'Layer 3'], ['#36D7B7', '#FFB74D', '#FF7676']):
    fig.add_trace(go.Bar(
        x=filtered_df.index,
        y=filtered_df[layer],
        name=f'{layer}',
        marker_color=color,
        opacity=0.7,
        legendgroup='bars'
    ))

# Add projections for protocols as lines
fig.add_trace(go.Scatter(
    x=total_chains_proj.index,
    y=total_chains_proj.values,
    name='Total Chains (Cum.)',
    line=dict(color='white', width=3),
    mode='lines',
    yaxis='y2',
    legendgroup='lines'
))

# Add protocol projections
fig.add_trace(go.Scatter(
    x=layerzero_proj.index,
    y=layerzero_proj.values,
    name='LayerZero (Cum.)',
    line=dict(color='#808080', width=2, dash='dot'),
    mode='lines',
    yaxis='y2',
    legendgroup='lines'
))

fig.add_trace(go.Scatter(
    x=hyperlane_proj.index,
    y=hyperlane_proj.values,
    name='Hyperlane (Cum.)',
    line=dict(color='#0000FF', width=2, dash='dot'),
    mode='lines',
    yaxis='y2',
    legendgroup='lines'
))

fig.add_trace(go.Scatter(
    x=wormhole_proj.index,
    y=wormhole_proj.values,
    name='Wormhole (Cum.)',
    line=dict(color='#800080', width=2, dash='dot'),
    mode='lines',
    yaxis='y2',
    legendgroup='lines'
))

# Add vertical line at 2025 to separate historical and projected data
fig.add_vline(x=2025, line_dash="dash", line_color="white", opacity=0.3)

# Update layout
fig.update_layout(
    title={
        'text': 'New Blockchain Platforms by Year and Projected Growth (2019-2027)',
        'y': 0.95,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': dict(size=20)
    },
    plot_bgcolor='rgb(17,17,17)',
    paper_bgcolor='rgb(17,17,17)',
    font=dict(color='white'),
    barmode='stack',
    xaxis=dict(
        title='Year',
        dtick=1,
        tickmode='linear',
        gridcolor='rgba(128,128,128,0.2)',
        tickcolor='white',
        range=[2018.5, 2027.5],
        showgrid=False
    ),
    yaxis=dict(
        title='Number of New Platforms',
        gridcolor='rgba(128,128,128,0.2)',
        tickcolor='white',
        showgrid=True,
        gridwidth=1,
        title_standoff=25,
        side='left',
        range=[0, max(filtered_df['Total New Chains']) * 1.1]
    ),
    yaxis2=dict(
        title='Cumulative Platforms',
        overlaying='y',
        side='right',
        showgrid=False,
        zeroline=False,
        tickcolor='white',
        title_standoff=25,
        range=[0, max(total_chains_proj.values) * 1.1]
    ),
    # Legend in top-left corner
    legend=dict(
        bgcolor='rgba(0,0,0,0.5)',  # Semi-transparent background
        font=dict(color='white', size=12),
        x=0.01,  # Position at the left edge
        y=0.99,  # Position at the top
        xanchor='left',
        yanchor='top',
        bordercolor='rgba(255,255,255,0.3)',
        borderwidth=1,
        itemsizing='constant',
        traceorder='grouped',
        tracegroupgap=5,
        groupclick='toggleitem'  # Allow clicking group names to toggle all items
    ),
    margin=dict(l=60, r=60, t=120, b=60),
    hovermode='x unified',
    height=800,
    width=1200,
    annotations=[
        dict(
            x=2025,
            y=filtered_df['Total New Chains'].max()/2,
            text="Projection Start",
            showarrow=True,
            arrowhead=1,
            ax=0,
            ay=-40,
            font=dict(color='white', size=14)
        ),
        # Add subtitle to clarify
        dict(
            x=0.5,
            y=1.08,
            xref='paper',
            yref='paper',
            text="Bars: New Annual Platforms by Layer | Lines: Cumulative Growth (Projections after 2025)",
            showarrow=False,
            font=dict(color='rgba(255,255,255,0.8)', size=14),
            align='center'
        )
    ]
)

# Adjust for better hover info
fig.update_traces(
    hovertemplate='<b>Year:</b> %{x}<br><b>%{data.name}:</b> %{y}<extra></extra>'
)

# Update bars to be more visually distinct
fig.update_layout(
    bargap=0.15,
    bargroupgap=0.1
)

# Display the figure
fig.show()

# Save as high-resolution PNG
fig.write_image("blockchain_combined_projection.png", width=1200, height=800)

In [14]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Get unique chains from each protocol through 2025
layerzero_chains = set(layerzero['Name'].unique())
wormhole_chains = set(wormhole['Name'].unique())
hyperlane_chains = set(hyperlane['Name'].unique())

# Get total unique chains across all protocols
all_provider_chains = layerzero_chains.union(wormhole_chains).union(hyperlane_chains)

# Calculate total number of chains by 2025
total_chains_2025 = int(df_combined['Cumulative Total Chains'].loc[2025])

# Calculate coverage metrics
layerzero_coverage = len(layerzero_chains)
wormhole_coverage = len(wormhole_chains)
hyperlane_coverage = len(hyperlane_chains)
combined_coverage = len(all_provider_chains)

# Calculate unserviced percentage
layerzero_unserviced_pct = 100 * (1 - layerzero_coverage / total_chains_2025)
wormhole_unserviced_pct = 100 * (1 - wormhole_coverage / total_chains_2025)
hyperlane_unserviced_pct = 100 * (1 - hyperlane_coverage / total_chains_2025)
combined_unserviced_pct = 100 * (1 - combined_coverage / total_chains_2025)

# Create a DataFrame for the charts
coverage_df = pd.DataFrame([
    {'Provider': 'LayerZero', 'Coverage %': 100 - layerzero_unserviced_pct, 'Unserviced %': layerzero_unserviced_pct},
    {'Provider': 'Wormhole', 'Coverage %': 100 - wormhole_unserviced_pct, 'Unserviced %': wormhole_unserviced_pct},
    {'Provider': 'Hyperlane', 'Coverage %': 100 - hyperlane_unserviced_pct, 'Unserviced %': hyperlane_unserviced_pct},
    {'Provider': 'Combined', 'Coverage %': 100 - combined_unserviced_pct, 'Unserviced %': combined_unserviced_pct},
])

# ================ STACKED BAR CHART (SEPARATE FIGURE) ================
fig_bar = go.Figure()

# Add bar chart for coverage comparison
fig_bar.add_trace(
    go.Bar(x=coverage_df['Provider'], y=coverage_df['Coverage %'], name='Coverage %', marker_color='#36D7B7')
)
fig_bar.add_trace(
    go.Bar(x=coverage_df['Provider'], y=coverage_df['Unserviced %'], name='Unserviced %', marker_color='#FFFF00')
)

# Add annotations to the bars
for i, provider in enumerate(coverage_df['Provider']):
    # Add annotation for the Coverage % (bottom part of the bar)
    coverage_value = coverage_df['Coverage %'].iloc[i]
    fig_bar.add_annotation(
        x=provider,
        y=coverage_value/2,  # Position in the middle of this section
        text=f"{coverage_value:.1f}%",
        showarrow=False,
        font=dict(color='black', size=12)
    )
    
    # Add annotation for the Unserviced % (top part of the bar)
    unserviced_value = coverage_df['Unserviced %'].iloc[i]
    fig_bar.add_annotation(
        x=provider,
        y=coverage_value + unserviced_value/2,  # Position in the middle of this section
        text=f"{unserviced_value:.1f}%",
        showarrow=False,
        font=dict(color='black', size=12)
    )

# Update layout for bar chart
fig_bar.update_layout(
    title={
        'text': 'Provider Coverage vs. Total Chains (Through 2025)',
        'y': 0.95,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': dict(size=20)
    },
    barmode='stack',
    plot_bgcolor='rgb(17,17,17)',
    paper_bgcolor='rgb(17,17,17)',
    font=dict(color='white'),
    legend=dict(
        bgcolor='rgba(0,0,0,0.5)',
        font=dict(color='white'),
        bordercolor='white',
        borderwidth=1
    ),
    height=600,
    width=800,
    yaxis=dict(
        title='Percentage (%)',
        gridcolor='rgba(128,128,128,0.2)',
        range=[0, 100],
        tickcolor='white'
    ),
    xaxis=dict(
        title='Provider',
        gridcolor='rgba(128,128,128,0.2)',
        tickcolor='white'
    ),
    annotations=[
        dict(
            x=0.5,
            y=1.08,
            xref='paper',
            yref='paper',
            text=f"Total Chains: {total_chains_2025} | Unserviced Market: {total_chains_2025 - combined_coverage} chains ({combined_unserviced_pct:.1f}%)",
            showarrow=False,
            font=dict(color='rgba(255,255,255,0.8)', size=14),
            align='center'
        )
    ]
)

# ================ PIE CHART (SEPARATE FIGURE) ================
fig_pie = go.Figure()

# Add pie chart for market share
fig_pie.add_trace(
    go.Pie(
        labels=['LayerZero', 'Wormhole', 'Hyperlane', 'Unserviced Market'],
        values=[layerzero_coverage, wormhole_coverage, hyperlane_coverage, total_chains_2025 - combined_coverage],
        marker_colors=['#808080', '#800080', '#0000FF', '#FFFF00'],
        hole=0.4,
        textinfo='label+percent',
        insidetextorientation='radial'
    )
)

# Update layout for pie chart
fig_pie.update_layout(
    title={
        'text': 'Market Share by Provider (Through 2025)',
        'y': 0.95,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': dict(size=20)
    },
    plot_bgcolor='rgb(17,17,17)',
    paper_bgcolor='rgb(17,17,17)',
    font=dict(color='white'),
    height=600,
    width=800,
    annotations=[
        dict(
            x=0.5,
            y=1.08,
            xref='paper',
            yref='paper',
            text=f"Note: This chart shows relative proportion including overlaps",
            showarrow=False,
            font=dict(color='rgba(255,255,255,0.8)', size=14),
            align='center'
        )
    ]
)

# Create years list (not range) for the time series
years_list = list(range(2021, 2026))

# Estimate cumulative coverage for each year
layerzero_by_year = [len(layerzero[layerzero['Year'] <= year]) for year in years_list]
wormhole_by_year = [len(wormhole[wormhole['Year'] <= year]) for year in years_list]
hyperlane_by_year = [len(hyperlane[hyperlane['Year'] <= year]) for year in years_list]

# Get total chains by year
total_chains_by_year = [df_combined['Cumulative Total Chains'].loc[year] if year in df_combined.index else np.nan for year in years_list]

# Calculate the combined unique coverage (accounting for overlap)
combined_coverage_by_year = []
for year in years_list:
    lz_chains = set(layerzero[layerzero['Year'] <= year]['Name'])
    wh_chains = set(wormhole[wormhole['Year'] <= year]['Name'])
    hp_chains = set(hyperlane[hyperlane['Year'] <= year]['Name'])
    combined_coverage_by_year.append(len(lz_chains.union(wh_chains).union(hp_chains)))

# Calculate unserviced percentage for each year
unserviced_pct_by_year = [100 * (1 - combined / total) for combined, total in zip(combined_coverage_by_year, total_chains_by_year)]

# ================ TIME SERIES FIGURE ================
fig2 = go.Figure()

# Add line for total chains
fig2.add_trace(go.Scatter(
    x=years_list,
    y=total_chains_by_year,
    name='Total Chains',
    line=dict(color='white', width=3),
    mode='lines+markers'
))

# Add lines for each provider
fig2.add_trace(go.Scatter(
    x=years_list,
    y=layerzero_by_year,
    name='LayerZero Coverage',
    line=dict(color='#808080', width=2),
    mode='lines+markers'
))

fig2.add_trace(go.Scatter(
    x=years_list,
    y=wormhole_by_year,
    name='Wormhole Coverage',
    line=dict(color='#800080', width=2),
    mode='lines+markers'
))

fig2.add_trace(go.Scatter(
    x=years_list,
    y=hyperlane_by_year,
    name='Hyperlane Coverage',
    line=dict(color='#0000FF', width=2),
    mode='lines+markers'
))

fig2.add_trace(go.Scatter(
    x=years_list,
    y=combined_coverage_by_year,
    name='Combined Coverage',
    line=dict(color='#36D7B7', width=2, dash='dot'),
    mode='lines+markers'
))

# Add a bar chart with the percentage of unserviced chains
fig2.add_trace(go.Bar(
    x=years_list,
    y=unserviced_pct_by_year,
    name='Unserviced Market %',
    marker_color='#FFFF00',
    opacity=0.7,
    yaxis='y2'
))

# Add annotations to the unserviced percentage bars
for i, year in enumerate(years_list):
    fig2.add_annotation(
        x=year,
        y=unserviced_pct_by_year[i],
        text=f"{unserviced_pct_by_year[i]:.1f}%",
        showarrow=False,
        yshift=10,
        font=dict(color='black', size=10),
        yref='y2'
    )

# Calculate the maximum value for the primary y-axis
max_y1 = max(max(total_chains_by_year), max(combined_coverage_by_year))

# Update layout for the second figure
fig2.update_layout(
    title={
        'text': 'Interoperability Provider Coverage Analysis (2021-2025)',
        'y': 0.95,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': dict(size=20)
    },
    plot_bgcolor='rgb(17,17,17)',
    paper_bgcolor='rgb(17,17,17)',
    font=dict(color='white'),
    legend=dict(
        bgcolor='rgba(0,0,0,0.5)',
        font=dict(color='white'),
        bordercolor='white',
        borderwidth=1
    ),
    height=600,
    width=1200,
    xaxis=dict(
        title='Year',
        dtick=1,
        tickmode='linear',
        gridcolor='rgba(128,128,128,0.2)',
        tickcolor='white'
    ),
    yaxis=dict(
        title='Number of Chains',
        gridcolor='rgba(128,128,128,0.2)',
        tickcolor='white',
        side='left',
        range=[0, max_y1 * 1.1]  # Ensure there's room at the top
    ),
    yaxis2=dict(
        title='Unserviced Market %',
        overlaying='y',
        side='right',
        range=[0, 100],  # Set fixed range from 0 to 100%
        tickcolor='white',
        gridcolor='rgba(128,128,128,0.2)'
    ),
    hovermode='x unified'
)

# Create a text annotation explaining the chart
fig2.add_annotation(
    x=0.5,
    y=1.08,
    xref='paper',
    yref='paper',
    text="Lines: Provider Chain Coverage | Yellow Bars: Unserviced Market Percentage",
    showarrow=False,
    font=dict(color='rgba(255,255,255,0.8)', size=14),
    align='center'
)

# ================ MARKET OPPORTUNITY FIGURE ================
fig3 = go.Figure()

# Calculate the market opportunity by year
market_opportunity = [total - combined for total, combined in zip(total_chains_by_year, combined_coverage_by_year)]

# Add line for unserviced chains
fig3.add_trace(go.Scatter(
    x=years_list,
    y=market_opportunity,
    name='Unserviced Chains',
    line=dict(color='white', width=3),
    mode='lines+markers'
))

# Add bar chart for the unserviced percentage
fig3.add_trace(go.Bar(
    x=years_list,
    y=unserviced_pct_by_year,
    name='Unserviced Market %',
    marker_color='#FFFF00',
    opacity=0.7,
    yaxis='y2'
))

# Add annotations to the unserviced percentage bars
for i, year in enumerate(years_list):
    fig3.add_annotation(
        x=year,
        y=unserviced_pct_by_year[i],
        text=f"{unserviced_pct_by_year[i]:.1f}%",
        showarrow=False,
        yshift=10,
        font=dict(color='black', size=10),
        yref='y2'
    )

# Calculate the maximum value for the primary y-axis
max_y1_fig3 = max(market_opportunity)

# Update layout for the third figure
fig3.update_layout(
    title={
        'text': 'Market Opportunity: Unserviced Chains (2021-2025)',
        'y': 0.95,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': dict(size=20)
    },
    plot_bgcolor='rgb(17,17,17)',
    paper_bgcolor='rgb(17,17,17)',
    font=dict(color='white'),
    legend=dict(
        bgcolor='rgba(0,0,0,0.5)',
        font=dict(color='white'),
        bordercolor='white',
        borderwidth=1
    ),
    height=600,
    width=1200,
    xaxis=dict(
        title='Year',
        dtick=1,
        tickmode='linear',
        gridcolor='rgba(128,128,128,0.2)',
        tickcolor='white'
    ),
    yaxis=dict(
        title='Number of Unserviced Chains',
        gridcolor='rgba(128,128,128,0.2)',
        tickcolor='white',
        side='left',
        range=[0, max_y1_fig3 * 1.1]
    ),
    yaxis2=dict(
        title='Unserviced Market %',
        overlaying='y',
        side='right',
        range=[0, 100],  # Set fixed range from 0 to 100%
        tickcolor='white',
        gridcolor='rgba(128,128,128,0.2)'
    ),
    hovermode='x unified',
    annotations=[
        dict(
            x=0.5,
            y=1.08,
            xref='paper',
            yref='paper',
            text="White Line: Unserviced Chains | Yellow Bars: Unserviced Market Percentage",
            showarrow=False,
            font=dict(color='rgba(255,255,255,0.8)', size=14),
            align='center'
        )
    ]
)

# Show the figures
fig_bar.show()
fig_pie.show()
fig2.show()
fig3.show()

# Save the figures
fig_bar.write_image("provider_coverage_bar.png", width=800, height=600)
fig_pie.write_image("provider_market_share_pie.png", width=800, height=600)
fig2.write_image("interoperability_coverage_trends.png", width=1200, height=600)
fig3.write_image("interoperability_market_opportunity.png", width=1200, height=600)

In [15]:
import pandas as pd
import plotly.graph_objects as go

# First, let's combine the blockchain layer data as before
year_counts_layer1 = layer1['Year'].value_counts().sort_index()
year_counts_layer2 = layer2['Year'].value_counts().sort_index()
year_counts_layer3 = layer3['Year'].value_counts().sort_index()

# Create cumulative blockchain counts
df_combined = pd.DataFrame({
    'Layer 1': year_counts_layer1,
    'Layer 2': year_counts_layer2,
    'Layer 3': year_counts_layer3
}).fillna(0)

df_combined = df_combined.sort_index()
df_combined['Total New Chains'] = df_combined.sum(axis=1)
df_combined['Cumulative Total Chains'] = df_combined['Total New Chains'].cumsum()

# Process each protocol's data separately
def get_protocol_cumulative(df):
    yearly_counts = df['Year'].value_counts().sort_index()
    return yearly_counts.cumsum()

layerzero_cumulative = get_protocol_cumulative(layerzero)
hyperlane_cumulative = get_protocol_cumulative(hyperlane)
wormhole_cumulative = get_protocol_cumulative(wormhole)

# Create shifted versions for plotting (adding 1 to each year)
df_shifted = pd.DataFrame()
df_shifted['Cumulative Total Chains'] = df_combined['Cumulative Total Chains']
df_shifted.index = df_combined.index + 1  # Shift years forward by 1

# Shift protocol integrations as well
layerzero_shifted = pd.Series(layerzero_cumulative.values, index=layerzero_cumulative.index + 1)
hyperlane_shifted = pd.Series(hyperlane_cumulative.values, index=hyperlane_cumulative.index + 1)
wormhole_shifted = pd.Series(wormhole_cumulative.values, index=wormhole_cumulative.index + 1)

# Create the plot
fig = go.Figure()

# Add cumulative blockchain count with shifted years
fig.add_trace(go.Scatter(
    x=df_shifted.index,
    y=df_shifted['Cumulative Total Chains'],
    name='Total Blockchain Platforms',
    line=dict(color='#36D7B7', width=4),
    mode='lines'
))

# Add each protocol's cumulative integrations with shifted years
fig.add_trace(go.Scatter(
    x=layerzero_shifted.index,
    y=layerzero_shifted.values,
    name='LayerZero Integrations',
    line=dict(color='#FF7676', width=2),
    mode='lines'
))

fig.add_trace(go.Scatter(
    x=hyperlane_shifted.index,
    y=hyperlane_shifted.values,
    name='Hyperlane Integrations',
    line=dict(color='#FFB74D', width=2),
    mode='lines'
))

fig.add_trace(go.Scatter(
    x=wormhole_shifted.index,
    y=wormhole_shifted.values,
    name='Wormhole Integrations',
    line=dict(color='#B39DDB', width=2),
    mode='lines'
))

# Update layout
fig.update_layout(
    title='Growth of Blockchain Platforms vs Individual Protocol Integrations (2014-2026)',
    xaxis_title='Year',
    yaxis_title='Cumulative Count',
    plot_bgcolor='rgb(17,17,17)',
    paper_bgcolor='rgb(17,17,17)',
    font=dict(color='white'),
    showlegend=True,
    legend=dict(
        bgcolor='rgba(0,0,0,0)',
        font=dict(color='white'),
        y=0.99,
        x=0.01,
        bordercolor='white',
        borderwidth=1
    ),
    xaxis=dict(
        gridcolor='rgba(128,128,128,0.2)',
        tickcolor='white',
        showgrid=False,
        range=[2014, 2026],  # Updated range to account for the shifted years
        dtick=1
    ),
    yaxis=dict(
        gridcolor='rgba(128,128,128,0.2)',
        tickcolor='white',
        showgrid=True,
        gridwidth=1
    ),
    hovermode='x unified'
)

# Add shaded area between total platforms and highest protocol line
max_protocol = pd.concat([
    layerzero_shifted,
    hyperlane_shifted,
    wormhole_shifted
]).groupby(level=0).max()

fig.add_trace(go.Scatter(
    x=df_shifted.index,
    y=df_shifted['Cumulative Total Chains'],
    fill='tonexty',
    fillcolor='rgba(54,215,183,0.1)',
    line=dict(width=0),
    showlegend=False,
    name='Integration Gap'
))

fig.show()

In [16]:
import pandas as pd
import plotly.graph_objects as go
import datetime

# First, let's combine the blockchain layer data as before
year_counts_layer1 = layer1['Year'].value_counts().sort_index()
year_counts_layer2 = layer2['Year'].value_counts().sort_index()
year_counts_layer3 = layer3['Year'].value_counts().sort_index()

# Create cumulative blockchain counts
df_combined = pd.DataFrame({
    'Layer 1': year_counts_layer1,
    'Layer 2': year_counts_layer2,
    'Layer 3': year_counts_layer3
}).fillna(0)

df_combined = df_combined.sort_index()
df_combined['Total New Chains'] = df_combined.sum(axis=1)
df_combined['Cumulative Total Chains'] = df_combined['Total New Chains'].cumsum()

# Get current month to determine how many months of 2025 have passed
current_month = datetime.datetime.now().month
months_elapsed_2025 = current_month
remaining_months_2025 = 12 - months_elapsed_2025

# Use 2024 data for projection
if 2024 in df_combined.index and months_elapsed_2025 > 0:
    # Calculate monthly rates based on 2024 total values
    monthly_rate_2024_layer1 = df_combined.loc[2024, 'Layer 1'] / 12
    monthly_rate_2024_layer2 = df_combined.loc[2024, 'Layer 2'] / 12
    monthly_rate_2024_layer3 = df_combined.loc[2024, 'Layer 3'] / 12
    
    # Calculate full year 2025 projections
    full_year_2025_layer1 = monthly_rate_2024_layer1 * 12
    full_year_2025_layer2 = monthly_rate_2024_layer2 * 12
    full_year_2025_layer3 = monthly_rate_2024_layer3 * 12
    
    # Update the 2025 projections in the dataframe
    if 2025 in df_combined.index:
        df_combined.loc[2025, 'Layer 1'] = full_year_2025_layer1
        df_combined.loc[2025, 'Layer 2'] = full_year_2025_layer2
        df_combined.loc[2025, 'Layer 3'] = full_year_2025_layer3
        df_combined.loc[2025, 'Total New Chains'] = full_year_2025_layer1 + full_year_2025_layer2 + full_year_2025_layer3
        # Recalculate the cumulative total
        df_combined['Cumulative Total Chains'] = df_combined['Total New Chains'].cumsum()

# Process each protocol's data separately
def get_protocol_cumulative(df):
    yearly_counts = df['Year'].value_counts().sort_index()
    return yearly_counts.cumsum()

layerzero_cumulative = get_protocol_cumulative(layerzero)
hyperlane_cumulative = get_protocol_cumulative(hyperlane)
wormhole_cumulative = get_protocol_cumulative(wormhole)

# Project the rest of 2025 for each protocol using 2024 data
def project_protocol_2025(cumulative_series, df_source):
    if 2024 in df_source['Year'].value_counts().index:
        count_2024 = df_source['Year'].value_counts().get(2024, 0)
        monthly_rate_2024 = count_2024 / 12
        
        # Calculate full year 2025 projection
        full_year_2025 = monthly_rate_2024 * 12
        
        # Get the value from 2024
        base_value = cumulative_series.get(2024, 0) if 2024 in cumulative_series.index else 0
        
        # Update the 2025 value
        if 2025 in cumulative_series.index:
            cumulative_series[2025] = base_value + full_year_2025
    
    return cumulative_series

layerzero_cumulative = project_protocol_2025(layerzero_cumulative, layerzero)
hyperlane_cumulative = project_protocol_2025(hyperlane_cumulative, hyperlane)
wormhole_cumulative = project_protocol_2025(wormhole_cumulative, wormhole)

# Create shifted versions for plotting (adding 1 to each year)
df_shifted = pd.DataFrame()
df_shifted['Cumulative Total Chains'] = df_combined['Cumulative Total Chains']
df_shifted.index = df_combined.index + 1  # Shift years forward by 1

# Shift protocol integrations as well
layerzero_shifted = pd.Series(layerzero_cumulative.values, index=layerzero_cumulative.index + 1)
hyperlane_shifted = pd.Series(hyperlane_cumulative.values, index=hyperlane_cumulative.index + 1)
wormhole_shifted = pd.Series(wormhole_cumulative.values, index=wormhole_cumulative.index + 1)

# Create the plot
fig = go.Figure()

# Add cumulative blockchain count with shifted years
fig.add_trace(go.Scatter(
    x=df_shifted.index,
    y=df_shifted['Cumulative Total Chains'],
    name='Total Blockchain Platforms',
    line=dict(color='#36D7B7', width=4),
    mode='lines'
))

# Add each protocol's cumulative integrations with shifted years
fig.add_trace(go.Scatter(
    x=layerzero_shifted.index,
    y=layerzero_shifted.values,
    name='LayerZero Integrations',
    line=dict(color='#FF7676', width=2),
    mode='lines'
))

fig.add_trace(go.Scatter(
    x=hyperlane_shifted.index,
    y=hyperlane_shifted.values,
    name='Hyperlane Integrations',
    line=dict(color='#FFB74D', width=2),
    mode='lines'
))

fig.add_trace(go.Scatter(
    x=wormhole_shifted.index,
    y=wormhole_shifted.values,
    name='Wormhole Integrations',
    line=dict(color='#B39DDB', width=2),
    mode='lines'
))

# Update layout
fig.update_layout(
    title='Growth of Blockchain Platforms vs Individual Protocol Integrations (2014-2026)',
    xaxis_title='Year',
    yaxis_title='Cumulative Count',
    plot_bgcolor='rgb(17,17,17)',
    paper_bgcolor='rgb(17,17,17)',
    font=dict(color='white'),
    showlegend=True,
    legend=dict(
        bgcolor='rgba(0,0,0,0)',
        font=dict(color='white'),
        y=0.99,
        x=0.01,
        bordercolor='white',
        borderwidth=1
    ),
    xaxis=dict(
        gridcolor='rgba(128,128,128,0.2)',
        tickcolor='white',
        showgrid=False,
        range=[2014, 2026],  # Updated range to account for the shifted years
        dtick=1
    ),
    yaxis=dict(
        gridcolor='rgba(128,128,128,0.2)',
        tickcolor='white',
        showgrid=True,
        gridwidth=1
    ),
    hovermode='x unified'
)

# Add shaded area between total platforms and highest protocol line
max_protocol = pd.concat([
    layerzero_shifted,
    hyperlane_shifted,
    wormhole_shifted
]).groupby(level=0).max()

fig.add_trace(go.Scatter(
    x=df_shifted.index,
    y=df_shifted['Cumulative Total Chains'],
    fill='tonexty',
    fillcolor='rgba(54,215,183,0.1)',
    line=dict(width=0),
    showlegend=False,
    name='Integration Gap'
))

fig.show()

In [45]:
import pandas as pd
import plotly.graph_objects as go
import datetime
import numpy as np
from scipy.optimize import curve_fit

# Function to fit exponential growth
def exp_growth(x, a, b):
    return a * np.exp(b * x)

# First, let's combine the blockchain layer data as before
year_counts_layer1 = layer1['Year'].value_counts().sort_index()
year_counts_layer2 = layer2['Year'].value_counts().sort_index()
year_counts_layer3 = layer3['Year'].value_counts().sort_index()

# Create cumulative blockchain counts
df_combined = pd.DataFrame({
    'Layer 1': year_counts_layer1,
    'Layer 2': year_counts_layer2,
    'Layer 3': year_counts_layer3
}).fillna(0)

df_combined = df_combined.sort_index()
df_combined['Total New Chains'] = df_combined.sum(axis=1)
df_combined['Cumulative Total Chains'] = df_combined['Total New Chains'].cumsum()

# Get current month to determine how many months of 2025 have passed
current_month = datetime.datetime.now().month
months_elapsed_2025 = current_month
remaining_months_2025 = 12 - months_elapsed_2025

# Use 2024 data for projection
if 2024 in df_combined.index and months_elapsed_2025 > 0:
    # Calculate monthly rates based on 2024 total values
    monthly_rate_2024_layer1 = df_combined.loc[2024, 'Layer 1'] / 12
    monthly_rate_2024_layer2 = df_combined.loc[2024, 'Layer 2'] / 12
    monthly_rate_2024_layer3 = df_combined.loc[2024, 'Layer 3'] / 12
    
    # Calculate full year 2025 projections
    full_year_2025_layer1 = monthly_rate_2024_layer1 * 12
    full_year_2025_layer2 = monthly_rate_2024_layer2 * 12
    full_year_2025_layer3 = monthly_rate_2024_layer3 * 12
    
    # Update the 2025 projections in the dataframe
    if 2025 in df_combined.index:
        df_combined.loc[2025, 'Layer 1'] = full_year_2025_layer1
        df_combined.loc[2025, 'Layer 2'] = full_year_2025_layer2
        df_combined.loc[2025, 'Layer 3'] = full_year_2025_layer3
        df_combined.loc[2025, 'Total New Chains'] = full_year_2025_layer1 + full_year_2025_layer2 + full_year_2025_layer3
        # Recalculate the cumulative total
        df_combined['Cumulative Total Chains'] = df_combined['Total New Chains'].cumsum()

# Process each protocol's data separately
def get_protocol_cumulative(df):
    yearly_counts = df['Year'].value_counts().sort_index()
    return yearly_counts.cumsum()

layerzero_cumulative = get_protocol_cumulative(layerzero)
hyperlane_cumulative = get_protocol_cumulative(hyperlane)
wormhole_cumulative = get_protocol_cumulative(wormhole)

# Project the rest of 2025 for each protocol using 2024 data
def project_protocol_2025(cumulative_series, df_source):
    if 2024 in df_source['Year'].value_counts().index:
        count_2024 = df_source['Year'].value_counts().get(2024, 0)
        monthly_rate_2024 = count_2024 / 12
        
        # Calculate full year 2025 projection
        full_year_2025 = monthly_rate_2024 * 12
        
        # Get the value from 2024
        base_value = cumulative_series.get(2024, 0) if 2024 in cumulative_series.index else 0
        
        # Update the 2025 value
        if 2025 in cumulative_series.index:
            cumulative_series[2025] = base_value + full_year_2025
    
    return cumulative_series

layerzero_cumulative = project_protocol_2025(layerzero_cumulative, layerzero)
hyperlane_cumulative = project_protocol_2025(hyperlane_cumulative, hyperlane)
wormhole_cumulative = project_protocol_2025(wormhole_cumulative, wormhole)

# Function to create exponential projection for 2026-2027
def create_exponential_projection(historical_series, projection_years=[2026, 2027]):
    # Extract data from 2020 onwards for fitting
    recent_data = historical_series[historical_series.index >= 2020] if len(historical_series[historical_series.index >= 2020]) > 2 else historical_series
    
    # Prepare data for curve fitting
    years = np.array(recent_data.index, dtype=float)
    values = np.array(recent_data.values, dtype=float)
    
    # Normalize years to avoid numerical issues
    years_normalized = years - min(years)
    
    # Fit exponential curve
    try:
        popt, _ = curve_fit(exp_growth, years_normalized, values, p0=[values[0], 0.1], maxfev=10000)
        a, b = popt
        
        # Project for future years
        projection_years_norm = np.array([year - min(years) for year in projection_years])
        projected_values = exp_growth(projection_years_norm, a, b)
        
        # Create a new series with projections
        projected_series = pd.Series(projected_values, index=projection_years)
        
        # Combine historical and projected data
        complete_series = pd.concat([historical_series, projected_series])
        
        return complete_series
    except:
        # Fallback to linear extrapolation if curve fitting fails
        if len(recent_data) >= 2:
            growth_rate = (recent_data.iloc[-1] - recent_data.iloc[-2]) * 1.1  # Add 10% to the last growth rate
            projected_series = pd.Series(
                [recent_data.iloc[-1] + growth_rate * i for i in range(1, len(projection_years) + 1)],
                index=projection_years
            )
            return pd.concat([historical_series, projected_series])
        else:
            return historical_series

# Create exponential projections for 2026-2027
df_combined_copy = df_combined.copy()
total_chains_projected = create_exponential_projection(df_combined_copy['Cumulative Total Chains'])
layerzero_projected = create_exponential_projection(layerzero_cumulative)
hyperlane_projected = create_exponential_projection(hyperlane_cumulative)
wormhole_projected = create_exponential_projection(wormhole_cumulative)

# Create shifted versions for plotting (adding 1 to each year)
df_shifted = pd.DataFrame()
df_shifted['Cumulative Total Chains'] = total_chains_projected
df_shifted.index = total_chains_projected.index + 1  # Shift years forward by 1

# Shift protocol integrations as well
layerzero_shifted = pd.Series(layerzero_projected.values, index=layerzero_projected.index + 1)
hyperlane_shifted = pd.Series(hyperlane_projected.values, index=hyperlane_projected.index + 1)
wormhole_shifted = pd.Series(wormhole_projected.values, index=wormhole_projected.index + 1)

# Filter data to show only 2019-2027
df_shifted_filtered = df_shifted[df_shifted.index.isin(range(2019, 2028))]
layerzero_shifted_filtered = layerzero_shifted[layerzero_shifted.index.isin(range(2019, 2028))]
hyperlane_shifted_filtered = hyperlane_shifted[hyperlane_shifted.index.isin(range(2019, 2028))]
wormhole_shifted_filtered = wormhole_shifted[wormhole_shifted.index.isin(range(2019, 2028))]

# Create the plot
fig = go.Figure()

# Add cumulative blockchain count with shifted years
fig.add_trace(go.Scatter(
    x=df_shifted_filtered.index,
    y=df_shifted_filtered['Cumulative Total Chains'],
    name='Total Blockchain Platforms',
    line=dict(color='#36D7B7', width=4),
    mode='lines'
))

# Add each protocol's cumulative integrations with shifted years
fig.add_trace(go.Scatter(
    x=layerzero_shifted_filtered.index,
    y=layerzero_shifted_filtered.values,
    name='LayerZero Integrations',
    line=dict(color='#FF7676', width=2),
    mode='lines'
))

fig.add_trace(go.Scatter(
    x=hyperlane_shifted_filtered.index,
    y=hyperlane_shifted_filtered.values,
    name='Hyperlane Integrations',
    line=dict(color='#FFB74D', width=2),
    mode='lines'
))

fig.add_trace(go.Scatter(
    x=wormhole_shifted_filtered.index,
    y=wormhole_shifted_filtered.values,
    name='Wormhole Integrations',
    line=dict(color='#B39DDB', width=2),
    mode='lines'
))

# Update layout
fig.update_layout(
    title='Growth of Blockchain Platforms vs Individual Protocol Integrations (2019-2027)',
    xaxis_title='Year',
    yaxis_title='Cumulative Count',
    plot_bgcolor='rgb(17,17,17)',
    paper_bgcolor='rgb(17,17,17)',
    font=dict(color='white'),
    showlegend=True,
    legend=dict(
        bgcolor='rgba(0,0,0,0)',
        font=dict(color='white'),
        y=0.99,
        x=0.01,
        bordercolor='white',
        borderwidth=1
    ),
    xaxis=dict(
        gridcolor='rgba(128,128,128,0.2)',
        tickcolor='white',
        showgrid=False,
        range=[2018.5, 2027.5],  # Focus on 2019-2027
        dtick=1
    ),
    yaxis=dict(
        gridcolor='rgba(128,128,128,0.2)',
        tickcolor='white',
        showgrid=True,
        gridwidth=1
    ),
    hovermode='x unified'
)

# Add shaded area between total platforms and highest protocol line
max_protocol_filtered = pd.concat([
    layerzero_shifted_filtered,
    hyperlane_shifted_filtered,
    wormhole_shifted_filtered
]).groupby(level=0).max()

fig.add_trace(go.Scatter(
    x=df_shifted_filtered.index,
    y=df_shifted_filtered['Cumulative Total Chains'],
    fill='tonexty',
    fillcolor='rgba(54,215,183,0.1)',
    line=dict(width=0),
    showlegend=False,
    name='Integration Gap'
))

# Add vertical line at 2025.5 (middle of 2025-2026 period) and annotation
fig.add_vline(
    x=2025.5, 
    line=dict(dash="dash", color="white", width=1.5), 
    opacity=0.7
)

# Get max y-value for annotation positioning
max_y = df_shifted_filtered['Cumulative Total Chains'].max()

fig.add_annotation(
    x=2025.5,
    y=max_y * 0.5,  # Position at middle height
    text="Projection Start",
    showarrow=True,
    arrowhead=2,
    arrowsize=1,
    arrowwidth=2,
    arrowcolor='white',
    ax=-60,
    ay=-40,
    font=dict(color='white', size=12),
    bordercolor='white',
    borderwidth=1,
    borderpad=4,
    bgcolor='rgba(17,17,17,0.7)'
)

fig.update_layout(
    title='Growth of Blockchain Platforms vs Individual Protocol Integrations (2019-2027)',
    xaxis_title='Year',
    yaxis_title='Cumulative Count',
    plot_bgcolor='white',        # <- figure background
    paper_bgcolor='white',       # <- surrounding “paper”
    font=dict(color='black'),    # text colour

    showlegend=True,
    legend=dict(
        bgcolor='rgba(0,0,0,0)',
        font=dict(color='black'),
        y=0.99, x=0.01,
        bordercolor='black',
        borderwidth=1
    ),
    xaxis=dict(
        gridcolor='rgba(0,0,0,0.1)',
        tickcolor='black',
        linecolor='black', showline=True,
        range=[2018.5, 2027.5],
        dtick=1
    ),
    yaxis=dict(
        gridcolor='rgba(0,0,0,0.1)',
        tickcolor='black',
        linecolor='black', showline=True,
        gridwidth=1
    ),
    hovermode='x unified'
)

# update projection marker colours so they remain visible on white
fig.update_shapes(dict(line=dict(color='black')))
fig.update_annotations(dict(font=dict(color='black'),
                            arrowcolor='black',
                            bordercolor='black',
                            bgcolor='rgba(255,255,255,0.8)'))


fig.show()

# Save as high-resolution PNG
fig.write_image("growth_blockchain_with_year_shifted_by_one.png", width=1200, height=800)

In [18]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np

# Create a DataFrame to store the growth rates
years = range(2019, 2028)
growth_data = pd.DataFrame(index=years)

# Calculate year-over-year growth rates for total blockchain platforms
growth_data['Total Platforms Count'] = df_shifted_filtered['Cumulative Total Chains']

# Add counts for each protocol
growth_data['LayerZero Count'] = layerzero_shifted_filtered
growth_data['Hyperlane Count'] = hyperlane_shifted_filtered
growth_data['Wormhole Count'] = wormhole_shifted_filtered

# Calculate year-over-year growth rates
for col in ['Total Platforms Count', 'LayerZero Count', 'Hyperlane Count', 'Wormhole Count']:
    growth_data[f'{col} Growth %'] = growth_data[col].pct_change() * 100

# Calculate integration coverage percentages
growth_data['LayerZero Coverage %'] = (growth_data['LayerZero Count'] / growth_data['Total Platforms Count']) * 100
growth_data['Hyperlane Coverage %'] = (growth_data['Hyperlane Count'] / growth_data['Total Platforms Count']) * 100
growth_data['Wormhole Coverage %'] = (growth_data['Wormhole Count'] / growth_data['Total Platforms Count']) * 100
growth_data['Combined Coverage %'] = ((growth_data[['LayerZero Count', 'Hyperlane Count', 'Wormhole Count']].max(axis=1)) / 
                                     growth_data['Total Platforms Count']) * 100

# Mark the projection start
growth_data['Projection'] = 'Historical'
growth_data.loc[2026:, 'Projection'] = 'Projected'

# Format the table for display
display_cols = [
    'Total Platforms Count', 'Total Platforms Count Growth %',
    'LayerZero Count', 'LayerZero Count Growth %', 'LayerZero Coverage %',
    'Hyperlane Count', 'Hyperlane Count Growth %', 'Hyperlane Coverage %',
    'Wormhole Count', 'Wormhole Count Growth %', 'Wormhole Coverage %',
    'Combined Coverage %',
    'Projection'
]

# Handle NaN values (for the first year's growth rate)
formatted_data = growth_data[display_cols].copy()
formatted_data.fillna('', inplace=True)

# Round numerical values for better display
for col in formatted_data.columns:
    if 'Count' in col and 'Growth' not in col:
        formatted_data[col] = formatted_data[col].apply(lambda x: f"{x:.0f}" if isinstance(x, (int, float)) else x)
    elif 'Growth' in col or 'Coverage' in col:
        formatted_data[col] = formatted_data[col].apply(lambda x: f"{x:.1f}%" if isinstance(x, (int, float)) else x)

# Create a Plotly table
fig = go.Figure(data=[go.Table(
    header=dict(
        values=['Year'] + list(formatted_data.columns),
        fill_color='#283442',
        font=dict(color='white', size=12),
        align='center',
        height=40
    ),
    cells=dict(
        values=[formatted_data.index] + [formatted_data[col] for col in formatted_data.columns],
        fill_color=[
            ['#283442' if i % 2 == 0 else '#1e2736' for i in range(len(formatted_data))],  # Year column
            ['#283442' if row % 2 == 0 else '#1e2736' for row in range(len(formatted_data))],  # Platform count
            ['#283442' if row % 2 == 0 else '#1e2736' for row in range(len(formatted_data))],  # Platform growth
            
            ['#283442' if row % 2 == 0 else '#1e2736' for row in range(len(formatted_data))],  # LayerZero count
            ['#283442' if row % 2 == 0 else '#1e2736' for row in range(len(formatted_data))],  # LayerZero growth
            ['#283442' if row % 2 == 0 else '#1e2736' for row in range(len(formatted_data))],  # LayerZero coverage
            
            ['#283442' if row % 2 == 0 else '#1e2736' for row in range(len(formatted_data))],  # Hyperlane count
            ['#283442' if row % 2 == 0 else '#1e2736' for row in range(len(formatted_data))],  # Hyperlane growth
            ['#283442' if row % 2 == 0 else '#1e2736' for row in range(len(formatted_data))],  # Hyperlane coverage
            
            ['#283442' if row % 2 == 0 else '#1e2736' for row in range(len(formatted_data))],  # Wormhole count
            ['#283442' if row % 2 == 0 else '#1e2736' for row in range(len(formatted_data))],  # Wormhole growth
            ['#283442' if row % 2 == 0 else '#1e2736' for row in range(len(formatted_data))],  # Wormhole coverage
            
            ['#283442' if row % 2 == 0 else '#1e2736' for row in range(len(formatted_data))],  # Combined coverage
            
            ['#283442' if val == 'Historical' else '#1e4736' if val == 'Partially Projected' else '#3a2e56'
             for val in growth_data['Projection']]  # Projection status with distinct colors
        ],
        font=dict(color='white', size=11),
        align=['center'] * (len(formatted_data.columns) + 1),
        height=30,
    )
)]
)

# Update layout
fig.update_layout(
    title='Blockchain Platform Growth vs. Protocol Integration Coverage (2019-2027)',
    font=dict(color='white'),
    paper_bgcolor='rgb(17,17,17)',
    margin=dict(l=0, r=0, t=50, b=0),
    height=400 + (len(formatted_data) * 30),  # Adjust height based on rows
    width=1200
)

fig.show()

# Also display a simplified version of the table focusing on key metrics
simplified_cols = [
    'Total Platforms Count', 'Total Platforms Count Growth %',
    'LayerZero Coverage %', 'Hyperlane Coverage %', 'Wormhole Coverage %',
    'Combined Coverage %', 'Projection'
]

simplified_table = formatted_data[simplified_cols].copy()

# Create a simplified Plotly table
fig2 = go.Figure(data=[go.Table(
    header=dict(
        values=['Year'] + list(simplified_table.columns),
        fill_color='#283442',
        font=dict(color='white', size=12),
        align='center',
        height=40
    ),
    cells=dict(
        values=[simplified_table.index] + [simplified_table[col] for col in simplified_table.columns],
        fill_color=[
            ['#283442' if i % 2 == 0 else '#1e2736' for i in range(len(simplified_table))],
            ['#283442' if row % 2 == 0 else '#1e2736' for row in range(len(simplified_table))],
            ['#283442' if row % 2 == 0 else '#1e2736' for row in range(len(simplified_table))],
            ['#283442' if row % 2 == 0 else '#1e2736' for row in range(len(simplified_table))],
            ['#283442' if row % 2 == 0 else '#1e2736' for row in range(len(simplified_table))],
            ['#283442' if row % 2 == 0 else '#1e2736' for row in range(len(simplified_table))],
            ['#283442' if row % 2 == 0 else '#1e2736' for row in range(len(simplified_table))],
            ['#283442' if val == 'Historical' else '#1e4736' if val == 'Partially Projected' else '#3a2e56'
             for val in growth_data['Projection']]
        ],
        font=dict(color='white', size=11),
        align=['center'] * (len(simplified_table.columns) + 1),
        height=30
    )
)]
)

# Update layout for simplified table
fig2.update_layout(
    title='Key Growth and Coverage Metrics (2019-2027)',
    font=dict(color='white'),
    paper_bgcolor='rgb(17,17,17)',
    margin=dict(l=0, r=0, t=50, b=0),
    height=400,
    width=1000
)

fig2.show()

# Save as high-resolution PNG
fig2.write_image("key_growth_coverage_metrics.png", width=1200, height=400)

/var/folders/7j/4shxr2bn0wq270wnn4zcvyxr0000gn/T/ipykernel_23309/857271256.py:44: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.



In [44]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np

# Create a DataFrame to store the growth rates
years = range(2019, 2028)
growth_data = pd.DataFrame(index=years)

# Calculate year-over-year growth rates for total blockchain platforms
growth_data['Total Platforms Count'] = df_shifted_filtered['Cumulative Total Chains']

# Add counts for each protocol
growth_data['LayerZero Count'] = layerzero_shifted_filtered
growth_data['Hyperlane Count'] = hyperlane_shifted_filtered
growth_data['Wormhole Count'] = wormhole_shifted_filtered

# Calculate year-over-year growth rates
for col in ['Total Platforms Count', 'LayerZero Count', 'Hyperlane Count', 'Wormhole Count']:
    growth_data[f'{col} Growth %'] = growth_data[col].pct_change() * 100

# Calculate integration coverage percentages
growth_data['LayerZero Coverage %'] = (growth_data['LayerZero Count'] / growth_data['Total Platforms Count']) * 100
growth_data['Hyperlane Coverage %'] = (growth_data['Hyperlane Count'] / growth_data['Total Platforms Count']) * 100
growth_data['Wormhole Coverage %'] = (growth_data['Wormhole Count'] / growth_data['Total Platforms Count']) * 100
growth_data['Combined Coverage %'] = ((growth_data[['LayerZero Count', 'Hyperlane Count', 'Wormhole Count']].max(axis=1)) / 
                                     growth_data['Total Platforms Count']) * 100

# Calculate unserviced market metrics
growth_data['Unserviced Platforms Count'] = growth_data['Total Platforms Count'] - growth_data[['LayerZero Count', 'Hyperlane Count', 'Wormhole Count']].max(axis=1)
growth_data['Unserviced Market %'] = 100 - growth_data['Combined Coverage %']
growth_data['Unserviced Market Growth %'] = growth_data['Unserviced Platforms Count'].pct_change() * 100

# Mark the projection start
growth_data['Projection'] = 'Historical'
growth_data.loc[2026:, 'Projection'] = 'Projected (Exponential)'
growth_data.loc[2025, 'Projection'] = 'Projected (Exponential)'

# Format the table for display
display_cols = [
    'Total Platforms Count', 'Total Platforms Count Growth %',
    'Combined Coverage %', 
    'Unserviced Platforms Count', 'Unserviced Market %', 'Unserviced Market Growth %',
    'Projection'
]

# Handle NaN values (for the first year's growth rate)
formatted_data = growth_data[display_cols].copy()
formatted_data.fillna('', inplace=True)

# Round numerical values for better display
for col in formatted_data.columns:
    if 'Count' in col and 'Growth' not in col:
        formatted_data[col] = formatted_data[col].apply(lambda x: f"{x:.0f}" if isinstance(x, (int, float)) else x)
    elif 'Growth' in col or 'Coverage' in col or 'Market' in col:
        formatted_data[col] = formatted_data[col].apply(lambda x: f"{x:.1f}%" if isinstance(x, (int, float)) else x)

# Create a Plotly table focusing on unserviced market
fig3 = go.Figure(data=[go.Table(
    header=dict(
        values=['Year'] + list(formatted_data.columns),
        fill_color='#283442',
        font=dict(color='white', size=12),
        align='center',
        height=40
    ),
    cells=dict(
        values=[formatted_data.index] + [formatted_data[col] for col in formatted_data.columns],
        fill_color=[
            ['#283442' if i % 2 == 0 else '#1e2736' for i in range(len(formatted_data))],  # Year column
            ['#283442' if row % 2 == 0 else '#1e2736' for row in range(len(formatted_data))],  # Total count
            ['#283442' if row % 2 == 0 else '#1e2736' for row in range(len(formatted_data))],  # Total growth
            ['#283442' if row % 2 == 0 else '#1e2736' for row in range(len(formatted_data))],  # Combined coverage
            ['#283442' if row % 2 == 0 else '#1e2736' for row in range(len(formatted_data))],  # Unserviced count
            ['#283442' if row % 2 == 0 else '#1e2736' for row in range(len(formatted_data))],  # Unserviced %
            ['#283442' if row % 2 == 0 else '#1e2736' for row in range(len(formatted_data))],  # Unserviced growth
            ['#283442' if val == 'Historical' else '#1e4736' if val == 'Partially Projected' else '#3a2e56'
             for val in growth_data['Projection']]  # Projection status
        ],
        font=dict(color='white', size=11),
        align=['center'] * (len(formatted_data.columns) + 1),
        height=30,
    )
)]
)

# Update layout
fig3.update_layout(
    title='Unserviced Blockchain Market Growth (2019-2027)',
    font=dict(color='white'),
    paper_bgcolor='rgb(17,17,17)',
    margin=dict(l=0, r=0, t=50, b=0),
    height=350,
    width=1200
)

# ---------- table colours ----------
row_stripe = ['white' if i % 2 == 0 else '#f7f7f7'   # light grey stripe
              for i in range(len(formatted_data))]

fig3 = go.Figure(data=[go.Table(
    header=dict(
        values=['Year'] + list(formatted_data.columns),
        fill_color='black',                 # black header
        font=dict(color='white', size=12),  # white header text
        align='center', height=40
    ),
    cells=dict(
        values=[formatted_data.index] +
               [formatted_data[col] for col in formatted_data.columns],
        fill_color=[row_stripe]*(len(formatted_data.columns)+1),  # body rows white/grey
        font=dict(color='black', size=11),   # black body text
        align=['center']*(len(formatted_data.columns)+1),
        height=30,
    )
)])

fig3.update_layout(
    title='Unserviced Blockchain Market Growth (2019-2027)',
    paper_bgcolor='white',   # white “paper”
    font=dict(color='black'),
    margin=dict(l=0, r=0, t=50, b=0),
    height=350, width=1200
)

# Also create a visualization of the unserviced market growth
fig4 = go.Figure()

# Add total platforms line
fig4.add_trace(go.Scatter(
    x=growth_data.index,
    y=growth_data['Total Platforms Count'],
    name='Total Blockchain Platforms',
    line=dict(color='#36D7B7', width=3),
    mode='lines'
))

# Add line for maximum serviced platforms
fig4.add_trace(go.Scatter(
    x=growth_data.index,
    y=growth_data[['LayerZero Count', 'Hyperlane Count', 'Wormhole Count']].max(axis=1),
    name='Max Protocol Coverage',
    line=dict(color='#FFB74D', width=3),
    mode='lines'
))

# Add filled area for unserviced market
fig4.add_trace(go.Scatter(
    x=growth_data.index,
    y=growth_data['Total Platforms Count'],
    name='Unserviced Market',
    fill='tonexty',
    fillcolor='rgba(255,118,118,0.3)',
    line=dict(width=0),
    mode='none'
))

# Add vertical line at projection start
fig4.add_vline(
    x=2025.5, 
    line=dict(dash="dash", color="white", width=1.5), 
    opacity=0.7
)

fig4.add_annotation(
    x=2025.5,
    y=growth_data['Total Platforms Count'].max() * 0.5,
    text="Projection Start",
    showarrow=True,
    arrowhead=2,
    arrowsize=1,
    arrowwidth=2,
    arrowcolor='white',
    ax=-40,
    ay=0,
    font=dict(color='white', size=12),
    bordercolor='white',
    borderwidth=1,
    borderpad=4,
    bgcolor='rgba(17,17,17,0.7)'
)

# Update layout
fig4.update_layout(
    title='Growth of Total Blockchain Platforms vs. Serviced Market (2019-2027)',
    xaxis_title='Year',
    yaxis_title='Number of Blockchain Platforms',
    plot_bgcolor='rgb(17,17,17)',
    paper_bgcolor='rgb(17,17,17)',
    font=dict(color='white'),
    showlegend=True,
    legend=dict(
        bgcolor='rgba(0,0,0,0)',
        font=dict(color='white'),
        y=0.99,
        x=0.01,
        bordercolor='white',
        borderwidth=1
    ),
    xaxis=dict(
        gridcolor='rgba(128,128,128,0.2)',
        tickcolor='white',
        showgrid=False,
        range=[2018.5, 2027.5],
        dtick=1
    ),
    yaxis=dict(
        gridcolor='rgba(128,128,128,0.2)',
        tickcolor='white',
        showgrid=True,
        gridwidth=1
    ),
    hovermode='x unified'
)

fig4.update_layout(
    title='Growth of Total Blockchain Platforms vs. Serviced Market (2019-2027)',
    xaxis_title='Year',
    yaxis_title='Number of Blockchain Platforms',
    plot_bgcolor='white',    # white plotting area
    paper_bgcolor='white',   # white surrounding “paper”
    font=dict(color='black'),
    legend=dict(
        bgcolor='rgba(0,0,0,0)',
        font=dict(color='black'),
        y=0.99, x=0.01,
        bordercolor='black',
        borderwidth=1
    ),
    xaxis=dict(
        gridcolor='rgba(0,0,0,0.1)',
        tickcolor='black',
        linecolor='black', showline=True,
        range=[2018.5, 2027.5],
        dtick=1
    ),
    yaxis=dict(
        gridcolor='rgba(0,0,0,0.1)',
        tickcolor='black',
        linecolor='black', showline=True,
        gridwidth=1
    ),
    hovermode='x unified'
)

# Make the projection marker dark so it’s visible on white
fig4.update_shapes(dict(line=dict(color='black')))
fig4.update_annotations(dict(font=dict(color='black'),
                             arrowcolor='black',
                             bordercolor='black',
                             bgcolor='rgba(255,255,255,0.8)'))

# Display the tables and chart
fig3.show()
fig4.show()

# Print a summary of the unserviced market growth
print("\nUnserviced Market Growth Summary:")
print("Year | Unserviced Platforms | % of Total | YoY Growth")
print("-" * 55)
for year in range(2019, 2028):
    unserviced_count = growth_data.loc[year, 'Unserviced Platforms Count']
    unserviced_pct = growth_data.loc[year, 'Unserviced Market %']
    growth_pct = growth_data.loc[year, 'Unserviced Market Growth %']
    growth_str = f"{growth_pct:.1f}%" if not pd.isna(growth_pct) else "N/A"
    projection = growth_data.loc[year, 'Projection']
    
    print(f"{year} | {unserviced_count:.0f} | {unserviced_pct:.1f}% | {growth_str} | {projection}")

# Save as high-resolution PNG
fig3.write_image("unserviced_market_growth.png", width=1500, height=500)
fig4.write_image("unserviced_market_visualization.png", width=1200, height=600)

/var/folders/7j/4shxr2bn0wq270wnn4zcvyxr0000gn/T/ipykernel_23309/2749999818.py:48: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.




Unserviced Market Growth Summary:
Year | Unserviced Platforms | % of Total | YoY Growth
-------------------------------------------------------
2019 | nan | nan% | N/A | Historical
2020 | nan | nan% | N/A | Historical
2021 | nan | nan% | N/A | Historical
2022 | 128 | 95.5% | N/A | Historical
2023 | 163 | 89.6% | 27.3% | Historical
2024 | 230 | 87.1% | 41.1% | Historical
2025 | 353 | 77.6% | 53.5% | Projected (Exponential)
2026 | 456 | 70.6% | 29.2% | Projected (Exponential)
2027 | 533 | 54.1% | 16.8% | Projected (Exponential)


In [20]:
# Compute average annual unserviced‐market growth rate over 2019–2027
avg_growth = growth_data['Unserviced Market Growth %'].dropna().mean()

print(f"Average annual unserviced market growth rate (2019–2027): {avg_growth:.2f}%")

Average annual unserviced market growth rate (2019–2027): 33.58%


In [46]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

# ── build cumulative DF (same as before) ────────────────────────────────────
df_cumulative = df_combined.copy()
for layer in ['Layer 1', 'Layer 2', 'Layer 3']:
    df_cumulative[layer] = df_cumulative[layer].cumsum()

df_cumulative['Total'] = df_cumulative[['Layer 1', 'Layer 2', 'Layer 3']].sum(axis=1)

# shift index +1 (still needed for alignment) -------------------------------
df_cumulative.index = df_cumulative.index + 1          # 2013→2014 … 2025→2026

# melt for the three layers --------------------------------------------------
df_melted = df_cumulative.reset_index(names=['Year']).melt(
    id_vars=['Year'],
    value_vars=['Layer 1', 'Layer 2', 'Layer 3'],
    var_name='Layer Type',
    value_name='Total Number of Platforms'
)

# base figure with layer lines ----------------------------------------------
fig = px.line(
    df_melted,
    x='Year',
    y='Total Number of Platforms',
    color='Layer Type',
    title='Cumulative Growth of Blockchain Platforms by Layer (From 2014 To End of 2025)',
    labels={'Year': 'Year', 'Total Number of Platforms': 'Total Platforms'},
    color_discrete_sequence=['#36D7B7', '#FFB74D', '#FF7676']
)

# add overall cumulative total ----------------------------------------------
fig.add_trace(
    go.Scatter(
        x=df_cumulative.index,
        y=df_cumulative['Total'],
        name='Cumulative Total',
        mode='lines',
        line=dict(color='#B39DDB', width=3, dash='dot')
    )
)

# dark theme layout tweaks ---------------------------------------------------
fig.update_layout(
    plot_bgcolor='white',          # figure (plot) background
    paper_bgcolor='white',         # surrounding “paper” background
    font=dict(color='black'),      # text colour

    yaxis=dict(
        gridcolor='rgba(0,0,0,0.1)',   # light grey horizontal gridlines
        showgrid=True,
        gridwidth=1
    ),
    legend=dict(
        bgcolor='rgba(0,0,0,0)',   # transparent legend background
        font=dict(color='black')
    )
)

# X-axis
fig.update_xaxes(
    showgrid=False,          # keep vertical gridlines off (or True if you want them)
    showline=True,           # draw the axis line
    linecolor='black',       # axis line colour
    linewidth=1,
    ticks='outside',         # draw tick marks
    tickcolor='black'
)

# Y-axis
fig.update_yaxes(
    showgrid=True,           # keep horizontal gridlines if you like
    gridcolor='rgba(0,0,0,0.1)',
    showline=True,           # draw the axis line
    linecolor='black',
    linewidth=1,
    ticks='outside',
    tickcolor='black'
)

fig.update_xaxes(showgrid=False)    # keep vertical gridlines off if you like     # <-- turn off vertical grid lines
fig.update_traces(mode='lines', line=dict(width=2))
fig.show()
fig.write_image("cumulative_growth_by_layer.png", width=1400, height=800, scale=2)